In [1]:
# ============================================================
# NOTEBOOK 02: FEATURE ENGINEERING
# Proyecto: Credit Risk Scoring ML
# Autor: Marín Serrato Barrios
# Descripción: Construcción de variables predictivas
#              para el modelo de scoring crediticio.
#              Basado en hallazgos del EDA (notebook 01).
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
import os

# Configuración
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

# Ruta base del proyecto (ruta absoluta, siempre funciona)
RUTA_PROYECTO = "c:/Users/Marin/Documents/PROYECTO ML_OPS/credit-risk-scoring-ml"
os.chdir(RUTA_PROYECTO)

# Ruta al dataset
RUTA_ZIP = "data/raw/home-credit-default-risk.zip"

print(f"Directorio de trabajo: {os.getcwd()}")
print(f"Dataset disponible:    {os.path.exists(RUTA_ZIP)}")
print("\n✅ Configuración correcta")

Directorio de trabajo: c:\Users\Marin\Documents\PROYECTO ML_OPS\credit-risk-scoring-ml
Dataset disponible:    True

✅ Configuración correcta


In [2]:
# --- CARGA DEL DATASET ---

# Cargamos application_train.csv igual que en el EDA
with zipfile.ZipFile(RUTA_ZIP, 'r') as z:
    with z.open('application_train.csv') as f:
        df = pd.read_csv(f)

print(f"Dataset cargado: {df.shape[0]:,} filas, {df.shape[1]:,} columnas")

# Separar TARGET y SK_ID_CURR del resto
# Los guardamos aparte para no mezclarlos con los features
TARGET = df['TARGET'].copy()
ID     = df['SK_ID_CURR'].copy()

print(f"TARGET separado:  {TARGET.shape[0]:,} valores")
print(f"Tasa de mora:     {TARGET.mean()*100:.2f}%")

Dataset cargado: 307,511 filas, 122 columnas
TARGET separado:  307,511 valores
Tasa de mora:     8.07%


In [3]:
# --- PASO 1: ELIMINAR VARIABLES DE BAJO VALOR ---

# Del EDA identificamos tres grupos a descartar:
#
# GRUPO 1: Variables de características del edificio
#          >50% de nulos y bajo poder predictivo
#
# GRUPO 2: FLAG_DOCUMENT con correlación casi cero
#
# GRUPO 3: Variables redundantes o de control

# GRUPO 1: Características del edificio
# Son todas las que terminan en _AVG, _MEDI, _MODE
# más algunas variables específicas del inmueble
cols_edificio = [col for col in df.columns
                 if col.endswith('_AVG')
                 or col.endswith('_MEDI')
                 or col.endswith('_MODE')]

print(f"Variables de edificio a eliminar: {len(cols_edificio)}")

# GRUPO 2: FLAG_DOCUMENT con correlación casi cero
# Del EDA: FLAG_DOCUMENT_5/7/10/12/19/20 ≈ 0.0002
# Por consistencia eliminamos todos excepto FLAG_DOCUMENT_3
# que sí tenía correlación relevante (0.0443)
cols_documentos_bajos = [
    'FLAG_DOCUMENT_2',  'FLAG_DOCUMENT_4',  'FLAG_DOCUMENT_5',
    'FLAG_DOCUMENT_6',  'FLAG_DOCUMENT_7',  'FLAG_DOCUMENT_8',
    'FLAG_DOCUMENT_9',  'FLAG_DOCUMENT_10', 'FLAG_DOCUMENT_11',
    'FLAG_DOCUMENT_12', 'FLAG_DOCUMENT_13', 'FLAG_DOCUMENT_14',
    'FLAG_DOCUMENT_15', 'FLAG_DOCUMENT_16', 'FLAG_DOCUMENT_17',
    'FLAG_DOCUMENT_18', 'FLAG_DOCUMENT_19', 'FLAG_DOCUMENT_20',
    'FLAG_DOCUMENT_21'
]

print(f"FLAG_DOCUMENT a eliminar:         {len(cols_documentos_bajos)}")

# GRUPO 3: Variables de consultas al Buró de muy baja frecuencia
# Hora y semana tienen correlación ≈ 0.0008
cols_buro_bajas = [
    'AMT_REQ_CREDIT_BUREAU_HOUR',
    'AMT_REQ_CREDIT_BUREAU_WEEK',
]

print(f"Consultas Buró a eliminar:        {len(cols_buro_bajas)}")

# Eliminar todas
cols_eliminar = cols_edificio + cols_documentos_bajos + cols_buro_bajas
df.drop(columns=cols_eliminar, inplace=True)

print(f"\nColumnas antes: 122")
print(f"Columnas después: {df.shape[1]}")
print(f"Variables eliminadas: {len(cols_eliminar)}")

Variables de edificio a eliminar: 47
FLAG_DOCUMENT a eliminar:         19
Consultas Buró a eliminar:        2

Columnas antes: 122
Columnas después: 54
Variables eliminadas: 68


In [4]:
# --- PASO 2 COMPLETO CORREGIDO: TRATAMIENTO DE VALORES FALTANTES ---
# Sintaxis correcta para pandas 3.0 (Copy-on-Write)
# Usar: df['col'] = df['col'].fillna(valor)
# NO usar: df['col'].fillna(valor, inplace=True)

print("ESTRATEGIA DE TRATAMIENTO DE MISSINGS:")
print("="*60)
print("LightGBM:   NaN preservados + FLAGS de ausencia")
print("Scorecard:  Imputación completa + FLAGS de ausencia")
print()

# Recrear los dos dataframes desde df limpio
# (el df que tiene 54 columnas después del paso 1)
df_lgbm      = df.copy()
df_scorecard = df.copy()

# ── PASO 2A: Crear FLAGS de ausencia (aplica a ambos) ─────
print("PASO 2A: Creando FLAGS de ausencia...")
print("-"*40)

# EXT_SOURCE_1: 56.4% nulos
df_lgbm['FLAG_EXT_SOURCE_1_NULO'] = (
    df_lgbm['EXT_SOURCE_1'].isna().astype(int))
df_scorecard['FLAG_EXT_SOURCE_1_NULO'] = (
    df_scorecard['EXT_SOURCE_1'].isna().astype(int))
print(f"FLAG_EXT_SOURCE_1_NULO: "
      f"{df_lgbm['FLAG_EXT_SOURCE_1_NULO'].sum():,} casos "
      f"({df_lgbm['FLAG_EXT_SOURCE_1_NULO'].mean()*100:.1f}%)")

# EXT_SOURCE_3: 19.8% nulos
df_lgbm['FLAG_EXT_SOURCE_3_NULO'] = (
    df_lgbm['EXT_SOURCE_3'].isna().astype(int))
df_scorecard['FLAG_EXT_SOURCE_3_NULO'] = (
    df_scorecard['EXT_SOURCE_3'].isna().astype(int))
print(f"FLAG_EXT_SOURCE_3_NULO: "
      f"{df_lgbm['FLAG_EXT_SOURCE_3_NULO'].sum():,} casos "
      f"({df_lgbm['FLAG_EXT_SOURCE_3_NULO'].mean()*100:.1f}%)")

# OCCUPATION_TYPE: 31.3% nulos
df_lgbm['FLAG_OCCUPATION_NULO'] = (
    df_lgbm['OCCUPATION_TYPE'].isna().astype(int))
df_scorecard['FLAG_OCCUPATION_NULO'] = (
    df_scorecard['OCCUPATION_TYPE'].isna().astype(int))
print(f"FLAG_OCCUPATION_NULO:   "
      f"{df_lgbm['FLAG_OCCUPATION_NULO'].sum():,} casos "
      f"({df_lgbm['FLAG_OCCUPATION_NULO'].mean()*100:.1f}%)")

# ── PASO 2B: Imputación SOLO para df_scorecard ────────────
print()
print("PASO 2B: Imputación para df_scorecard...")
print("-"*40)

# Calcular medianas ANTES de imputar
mediana_ext1   = df_scorecard['EXT_SOURCE_1'].median()
mediana_ext2   = df_scorecard['EXT_SOURCE_2'].median()
mediana_ext3   = df_scorecard['EXT_SOURCE_3'].median()
moda_suite     = df_scorecard['NAME_TYPE_SUITE'].mode()[0]

# EXT_SOURCE_1
df_scorecard['EXT_SOURCE_1'] = (
    df_scorecard['EXT_SOURCE_1'].fillna(mediana_ext1))
print(f"EXT_SOURCE_1    → mediana: {mediana_ext1:.4f}")

# EXT_SOURCE_2
df_scorecard['EXT_SOURCE_2'] = (
    df_scorecard['EXT_SOURCE_2'].fillna(mediana_ext2))
print(f"EXT_SOURCE_2    → mediana: {mediana_ext2:.4f}")

# EXT_SOURCE_3
df_scorecard['EXT_SOURCE_3'] = (
    df_scorecard['EXT_SOURCE_3'].fillna(mediana_ext3))
print(f"EXT_SOURCE_3    → mediana: {mediana_ext3:.4f}")

# OCCUPATION_TYPE: categoría propia 'Unknown'
df_scorecard['OCCUPATION_TYPE'] = (
    df_scorecard['OCCUPATION_TYPE'].fillna('Unknown'))
print(f"OCCUPATION_TYPE → 'Unknown'")

# NAME_TYPE_SUITE: moda
df_scorecard['NAME_TYPE_SUITE'] = (
    df_scorecard['NAME_TYPE_SUITE'].fillna(moda_suite))
print(f"NAME_TYPE_SUITE → moda: '{moda_suite}'")

# Variables numéricas con pocos nulos
cols_mediana = ['AMT_ANNUITY', 'AMT_GOODS_PRICE', 'CNT_FAM_MEMBERS']
for col in cols_mediana:
    nulos = df_scorecard[col].isna().sum()
    if nulos > 0:
        mediana = df_scorecard[col].median()
        df_scorecard[col] = df_scorecard[col].fillna(mediana)
        print(f"{col:<20} → mediana: {mediana:.0f} ({nulos} casos)")

# ── PASO 2C: Imputación mínima para df_lgbm ───────────────
# LightGBM maneja NaN nativamente en EXT_SOURCE y OCCUPATION
# Solo imputamos variables con muy pocos nulos donde
# el NaN no aporta información
print()
print("PASO 2C: Imputación mínima para df_lgbm...")
print("-"*40)

for col in cols_mediana:
    nulos = df_lgbm[col].isna().sum()
    if nulos > 0:
        mediana = df_lgbm[col].median()
        df_lgbm[col] = df_lgbm[col].fillna(mediana)
        print(f"{col:<20} → mediana: {mediana:.0f} ({nulos} casos)")

# ── VERIFICACIÓN FINAL ────────────────────────────────────
print()
print("VERIFICACIÓN FINAL:")
print("="*60)

nulos_lgbm  = df_lgbm.isnull().sum()
nulos_score = df_scorecard.isnull().sum()

vars_nan_lgbm  = nulos_lgbm[nulos_lgbm > 0]
vars_nan_score = nulos_score[nulos_score > 0]

print(f"df_lgbm      - variables con NaN: {len(vars_nan_lgbm)}")
if len(vars_nan_lgbm) > 0:
    for col, n in vars_nan_lgbm.items():
        print(f"  {col}: {n:,} NaN  ← LightGBM los maneja nativamente")

print()
print(f"df_scorecard - variables con NaN: {len(vars_nan_score)} ← debe ser 0")
if len(vars_nan_score) > 0:
    print("  PROBLEMA: aún hay NaN en scorecard")
    print(vars_nan_score.to_string())
else:
    print("  ✅ Sin NaN, listo para regresión logística")

print()
print(f"Shape df_lgbm:      {df_lgbm.shape}")
print(f"Shape df_scorecard: {df_scorecard.shape}")

ESTRATEGIA DE TRATAMIENTO DE MISSINGS:
LightGBM:   NaN preservados + FLAGS de ausencia
Scorecard:  Imputación completa + FLAGS de ausencia

PASO 2A: Creando FLAGS de ausencia...
----------------------------------------
FLAG_EXT_SOURCE_1_NULO: 173,378 casos (56.4%)
FLAG_EXT_SOURCE_3_NULO: 60,965 casos (19.8%)
FLAG_OCCUPATION_NULO:   96,391 casos (31.3%)

PASO 2B: Imputación para df_scorecard...
----------------------------------------
EXT_SOURCE_1    → mediana: 0.5060
EXT_SOURCE_2    → mediana: 0.5660
EXT_SOURCE_3    → mediana: 0.5353
OCCUPATION_TYPE → 'Unknown'
NAME_TYPE_SUITE → moda: 'Unaccompanied'
AMT_ANNUITY          → mediana: 24903 (12 casos)
AMT_GOODS_PRICE      → mediana: 450000 (278 casos)
CNT_FAM_MEMBERS      → mediana: 2 (2 casos)

PASO 2C: Imputación mínima para df_lgbm...
----------------------------------------
AMT_ANNUITY          → mediana: 24903 (12 casos)
AMT_GOODS_PRICE      → mediana: 450000 (278 casos)
CNT_FAM_MEMBERS      → mediana: 2 (2 casos)

VERIFICACIÓN FINAL

In [5]:
# --- PASO 2D: Imputación de variables pendientes en df_scorecard ---

print("PASO 2D: Imputando variables pendientes en df_scorecard...")
print("-"*40)

# OWN_CAR_AGE: 202,929 nulos (65.9%)
# Los nulos son clientes SIN auto (FLAG_OWN_CAR = 'N')
# No tiene sentido imputar con mediana de los que SÍ tienen auto
# Imputamos con 0 → "no tiene auto, antigüedad = 0"
df_scorecard['OWN_CAR_AGE'] = df_scorecard['OWN_CAR_AGE'].fillna(0)
print(f"OWN_CAR_AGE              → 0 (sin auto)")

# Variables del círculo social
# OBS/DEF_30/60_CNT_SOCIAL_CIRCLE: 1,021 nulos
# Observaciones y defaults en el círculo social del cliente
# Imputamos con 0 → sin observaciones registradas
cols_social = ['OBS_30_CNT_SOCIAL_CIRCLE', 'DEF_30_CNT_SOCIAL_CIRCLE',
               'OBS_60_CNT_SOCIAL_CIRCLE', 'DEF_60_CNT_SOCIAL_CIRCLE']
for col in cols_social:
    df_scorecard[col] = df_scorecard[col].fillna(0)
    print(f"{col:<30} → 0")

# DAYS_LAST_PHONE_CHANGE: solo 1 nulo
# Imputamos con mediana
mediana_phone = df_scorecard['DAYS_LAST_PHONE_CHANGE'].median()
df_scorecard['DAYS_LAST_PHONE_CHANGE'] = (
    df_scorecard['DAYS_LAST_PHONE_CHANGE'].fillna(mediana_phone))
print(f"DAYS_LAST_PHONE_CHANGE   → mediana: {mediana_phone:.0f}")

# AMT_REQ_CREDIT_BUREAU_DAY/MON/QRT/YEAR: 41,519 nulos
# Son consultas al buró en distintos periodos
# Nulos probablemente = sin consultas recientes
# Imputamos con 0 → sin consultas
cols_buro = ['AMT_REQ_CREDIT_BUREAU_DAY',
             'AMT_REQ_CREDIT_BUREAU_MON',
             'AMT_REQ_CREDIT_BUREAU_QRT',
             'AMT_REQ_CREDIT_BUREAU_YEAR']
for col in cols_buro:
    df_scorecard[col] = df_scorecard[col].fillna(0)
    print(f"{col:<30} → 0")

# VERIFICACIÓN FINAL
print()
print("VERIFICACIÓN FINAL COMPLETA:")
print("="*60)
nulos_score = df_scorecard.isnull().sum()
vars_nan_score = nulos_score[nulos_score > 0]

print(f"df_scorecard - variables con NaN: {len(vars_nan_score)}", end=" ")
if len(vars_nan_score) == 0:
    print("✅ LISTO")
else:
    print("❌ PROBLEMA")
    print(vars_nan_score.to_string())

print(f"\ndf_lgbm      - variables con NaN: "
      f"{(df_lgbm.isnull().sum() > 0).sum()} "
      f"(manejados nativamente por LightGBM) ✅")

print(f"\nShape df_lgbm:      {df_lgbm.shape}")
print(f"Shape df_scorecard: {df_scorecard.shape}")

PASO 2D: Imputando variables pendientes en df_scorecard...
----------------------------------------
OWN_CAR_AGE              → 0 (sin auto)
OBS_30_CNT_SOCIAL_CIRCLE       → 0
DEF_30_CNT_SOCIAL_CIRCLE       → 0
OBS_60_CNT_SOCIAL_CIRCLE       → 0
DEF_60_CNT_SOCIAL_CIRCLE       → 0
DAYS_LAST_PHONE_CHANGE   → mediana: -757
AMT_REQ_CREDIT_BUREAU_DAY      → 0
AMT_REQ_CREDIT_BUREAU_MON      → 0
AMT_REQ_CREDIT_BUREAU_QRT      → 0
AMT_REQ_CREDIT_BUREAU_YEAR     → 0

VERIFICACIÓN FINAL COMPLETA:
df_scorecard - variables con NaN: 0 ✅ LISTO

df_lgbm      - variables con NaN: 15 (manejados nativamente por LightGBM) ✅

Shape df_lgbm:      (307511, 57)
Shape df_scorecard: (307511, 57)


In [6]:
# --- PASO 3: CONSTRUCCIÓN DE FEATURES DERIVADOS ---

# Aquí transformamos variables crudas en variables
# con significado de negocio directo.
# Estos features capturan relaciones que el modelo
# no podría descubrir fácilmente por sí solo.

print("PASO 3: CONSTRUCCIÓN DE FEATURES DERIVADOS")
print("="*60)

# ── FEATURE 1: DTI (Debt-to-Income Ratio) ─────────────────
# Vimos en el EDA que AMT_INCOME_TOTAL es MENSUAL
# AMT_ANNUITY también es mensual
# DTI = pago mensual / ingreso mensual

for dataset in [df_lgbm, df_scorecard]:
    dataset['DTI'] = dataset['AMT_ANNUITY'] / dataset['AMT_INCOME_TOTAL']

print(f"DTI:")
print(f"  Media:   {df_lgbm['DTI'].mean():.4f} ({df_lgbm['DTI'].mean()*100:.1f}%)")
print(f"  Mediana: {df_lgbm['DTI'].median():.4f} ({df_lgbm['DTI'].median()*100:.1f}%)")

# ── FEATURE 2: RATIO CRÉDITO / INGRESO ANUAL ──────────────
# Cuántos meses de ingreso representa el crédito total
# Mide el tamaño relativo del crédito vs capacidad de pago

for dataset in [df_lgbm, df_scorecard]:
    dataset['RATIO_CREDITO_INGRESO'] = (
        dataset['AMT_CREDIT'] / dataset['AMT_INCOME_TOTAL'])

print(f"\nRATIO_CREDITO_INGRESO:")
print(f"  Media:   {df_lgbm['RATIO_CREDITO_INGRESO'].mean():.2f} meses de ingreso")
print(f"  Mediana: {df_lgbm['RATIO_CREDITO_INGRESO'].median():.2f} meses de ingreso")

# ── FEATURE 3: PLAZO ESTIMADO DEL CRÉDITO ─────────────────
# AMT_CREDIT / AMT_ANNUITY = número de meses del crédito
# Vimos en el EDA que esto da ~20 meses en promedio

for dataset in [df_lgbm, df_scorecard]:
    dataset['PLAZO_MESES'] = (
        dataset['AMT_CREDIT'] / dataset['AMT_ANNUITY'])

print(f"\nPLAZO_MESES:")
print(f"  Media:   {df_lgbm['PLAZO_MESES'].mean():.1f} meses")
print(f"  Mediana: {df_lgbm['PLAZO_MESES'].median():.1f} meses")

# ── FEATURE 4: EDAD EN AÑOS ────────────────────────────────
# Ya lo calculamos en el EDA, lo incluimos formalmente

for dataset in [df_lgbm, df_scorecard]:
    dataset['EDAD_AÑOS'] = (-dataset['DAYS_BIRTH'] / 365).astype(int)

print(f"\nEDAD_AÑOS:")
print(f"  Media:   {df_lgbm['EDAD_AÑOS'].mean():.1f} años")
print(f"  Mediana: {df_lgbm['EDAD_AÑOS'].median():.1f} años")

# ── FEATURE 5: ANTIGÜEDAD LABORAL ─────────────────────────
# Tratar valor centinela 365243 (pensionados/desempleados)
# y convertir a años positivos

for dataset in [df_lgbm, df_scorecard]:
    # Reemplazar centinela con NaN
    days_employed_limpio = dataset['DAYS_EMPLOYED'].replace(365243, np.nan)
    # Convertir a años positivos
    dataset['ANTIGUEDAD_LABORAL_AÑOS'] = (-days_employed_limpio / 365)
    # Flag de pensionado/desempleado
    dataset['FLAG_PENSIONADO_DESEMPLEADO'] = (
        dataset['DAYS_EMPLOYED'] == 365243).astype(int)

# Para scorecard: imputar NaN de antigüedad con 0
df_scorecard['ANTIGUEDAD_LABORAL_AÑOS'] = (
    df_scorecard['ANTIGUEDAD_LABORAL_AÑOS'].fillna(0))

print(f"\nANTIGUEDAD_LABORAL_AÑOS:")
print(f"  Media:   {df_lgbm['ANTIGUEDAD_LABORAL_AÑOS'].mean():.1f} años")
print(f"  Mediana: {df_lgbm['ANTIGUEDAD_LABORAL_AÑOS'].median():.1f} años")
print(f"  Pensionados/desempleados: "
      f"{df_lgbm['FLAG_PENSIONADO_DESEMPLEADO'].sum():,}")

# ── FEATURE 6: EXT_SOURCE PROMEDIO ────────────────────────
# Combinar los tres scores externos en un promedio
# Captura el perfil crediticio general del cliente

for dataset in [df_lgbm, df_scorecard]:
    dataset['EXT_SOURCE_PROMEDIO'] = (
        dataset[['EXT_SOURCE_1',
                 'EXT_SOURCE_2',
                 'EXT_SOURCE_3']].mean(axis=1))

print(f"\nEXT_SOURCE_PROMEDIO:")
print(f"  Media:   {df_lgbm['EXT_SOURCE_PROMEDIO'].mean():.4f}")
print(f"  Mediana: {df_lgbm['EXT_SOURCE_PROMEDIO'].median():.4f}")

# Diferencia buenos vs malos
media_buenos = df_lgbm[df_lgbm['TARGET']==0]['EXT_SOURCE_PROMEDIO'].mean()
media_malos  = df_lgbm[df_lgbm['TARGET']==1]['EXT_SOURCE_PROMEDIO'].mean()
print(f"  Buenos:  {media_buenos:.4f}")
print(f"  Malos:   {media_malos:.4f}")
print(f"  Diferencia: {media_buenos-media_malos:.4f}")

# ── FEATURE 7: EXT_SOURCE MÍNIMO ──────────────────────────
# El score más bajo de los tres
# Captura el peor antecedente crediticio

for dataset in [df_lgbm, df_scorecard]:
    dataset['EXT_SOURCE_MIN'] = (
        dataset[['EXT_SOURCE_1',
                 'EXT_SOURCE_2',
                 'EXT_SOURCE_3']].min(axis=1))

print(f"\nEXT_SOURCE_MIN (peor score):")
media_buenos = df_lgbm[df_lgbm['TARGET']==0]['EXT_SOURCE_MIN'].mean()
media_malos  = df_lgbm[df_lgbm['TARGET']==1]['EXT_SOURCE_MIN'].mean()
print(f"  Buenos:     {media_buenos:.4f}")
print(f"  Malos:      {media_malos:.4f}")
print(f"  Diferencia: {media_buenos-media_malos:.4f}")

# ── RESUMEN ────────────────────────────────────────────────
print()
print("FEATURES DERIVADOS CREADOS:")
print("="*60)
features_nuevos = ['DTI', 'RATIO_CREDITO_INGRESO', 'PLAZO_MESES',
                   'EDAD_AÑOS', 'ANTIGUEDAD_LABORAL_AÑOS',
                   'FLAG_PENSIONADO_DESEMPLEADO',
                   'EXT_SOURCE_PROMEDIO', 'EXT_SOURCE_MIN']
for f in features_nuevos:
    print(f"  ✅ {f}")

print(f"\nShape df_lgbm:      {df_lgbm.shape}")
print(f"Shape df_scorecard: {df_scorecard.shape}")

PASO 3: CONSTRUCCIÓN DE FEATURES DERIVADOS
DTI:
  Media:   0.1809 (18.1%)
  Mediana: 0.1628 (16.3%)

RATIO_CREDITO_INGRESO:
  Media:   3.96 meses de ingreso
  Mediana: 3.27 meses de ingreso

PLAZO_MESES:
  Media:   21.6 meses
  Mediana: 20.0 meses

EDAD_AÑOS:
  Media:   43.4 años
  Mediana: 43.0 años

ANTIGUEDAD_LABORAL_AÑOS:
  Media:   6.5 años
  Mediana: 4.5 años
  Pensionados/desempleados: 55,374

EXT_SOURCE_PROMEDIO:
  Media:   0.5093
  Mediana: 0.5245
  Buenos:  0.5191
  Malos:   0.3970
  Diferencia: 0.1221

EXT_SOURCE_MIN (peor score):
  Buenos:     0.4099
  Malos:      0.2824
  Diferencia: 0.1275

FEATURES DERIVADOS CREADOS:
  ✅ DTI
  ✅ RATIO_CREDITO_INGRESO
  ✅ PLAZO_MESES
  ✅ EDAD_AÑOS
  ✅ ANTIGUEDAD_LABORAL_AÑOS
  ✅ FLAG_PENSIONADO_DESEMPLEADO
  ✅ EXT_SOURCE_PROMEDIO
  ✅ EXT_SOURCE_MIN

Shape df_lgbm:      (307511, 65)
Shape df_scorecard: (307511, 65)
